<a href="https://colab.research.google.com/github/Poorvi-M/Sleep-Health-Classifier/blob/main/Sleep_Health.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Importing libraries and data.**

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohankrishnathalla/sleep-health-and-daily-performance-dataset")

print("Path to dataset files:", path)


Using Colab cache for faster access to the 'sleep-health-and-daily-performance-dataset' dataset.
Path to dataset files: /kaggle/input/sleep-health-and-daily-performance-dataset


In [2]:
import pandas as pd
import os

files = os.listdir(path)
print("Files found:", files)

df = pd.read_csv(path + "/" + files[0])  # path is already a variable from cell 1

print("Shape:", df.shape)
print("\nColumn names:\n", df.columns.tolist())
print("\nFirst 5 rows:")
df.head()

Files found: ['sleep_health_dataset.csv']
Shape: (100000, 32)

Column names:
 ['person_id', 'age', 'gender', 'occupation', 'bmi', 'country', 'sleep_duration_hrs', 'sleep_quality_score', 'rem_percentage', 'deep_sleep_percentage', 'sleep_latency_mins', 'wake_episodes_per_night', 'caffeine_mg_before_bed', 'alcohol_units_before_bed', 'screen_time_before_bed_mins', 'exercise_day', 'steps_that_day', 'nap_duration_mins', 'stress_score', 'work_hours_that_day', 'chronotype', 'mental_health_condition', 'heart_rate_resting_bpm', 'sleep_aid_used', 'shift_work', 'room_temperature_celsius', 'weekend_sleep_diff_hrs', 'season', 'day_type', 'cognitive_performance_score', 'sleep_disorder_risk', 'felt_rested']

First 5 rows:


,person_id,age,gender,occupation,bmi,country,sleep_duration_hrs,sleep_quality_score,rem_percentage,deep_sleep_percentage,...,heart_rate_resting_bpm,sleep_aid_used,shift_work,room_temperature_celsius,weekend_sleep_diff_hrs,season,day_type,cognitive_performance_score,sleep_disorder_risk,felt_rested
0,1,29,Female,Driver,25.7,Japan,6.19,6.6,22.5,19.3,...,63,0,0,20.1,1.84,Autumn,Weekday,73.4,Healthy,0
1,2,55,Female,Software Engineer,22.0,USA,8.32,6.9,26.9,14.9,...,52,1,0,18.0,0.13,Winter,Weekend,99.4,Healthy,1
2,3,42,Male,Nurse,25.0,India,3.74,1.0,20.2,16.2,...,72,0,1,17.9,1.67,Spring,Weekend,2.5,Severe,0
3,4,37,Female,Student,29.5,India,6.79,6.4,17.7,17.7,...,71,0,0,19.1,2.37,Summer,Weekend,67.8,Healthy,0
4,5,23,Male,Lawyer,23.6,Spain,5.02,3.2,23.3,18.3,...,71,0,0,19.7,1.26,Summer,Weekday,38.1,Mild,0


In [3]:
print(df['sleep_disorder_risk'].value_counts())
print()
print("Number of missing values:")
print(df.isnull().sum())
print("Column data types:")
print(df.dtypes)

sleep_disorder_risk
Healthy     54156
Mild        33479
Moderate     8299
Severe       4066
Name: count, dtype: int64

Number of missing values:
person_id                      0
age                            0
gender                         0
occupation                     0
bmi                            0
country                        0
sleep_duration_hrs             0
sleep_quality_score            0
rem_percentage                 0
deep_sleep_percentage          0
sleep_latency_mins             0
wake_episodes_per_night        0
caffeine_mg_before_bed         0
alcohol_units_before_bed       0
screen_time_before_bed_mins    0
exercise_day                   0
steps_that_day                 0
nap_duration_mins              0
stress_score                   0
work_hours_that_day            0
chronotype                     0
mental_health_condition        0
heart_rate_resting_bpm         0
sleep_aid_used                 0
shift_work                     0
room_temperature_celsius      

In [4]:
df= df.drop(['person_id'], axis=1)
df.select_dtypes('object')
df.head()
print(df.shape)
print(df.columns.tolist())

(100000, 31)
['age', 'gender', 'occupation', 'bmi', 'country', 'sleep_duration_hrs', 'sleep_quality_score', 'rem_percentage', 'deep_sleep_percentage', 'sleep_latency_mins', 'wake_episodes_per_night', 'caffeine_mg_before_bed', 'alcohol_units_before_bed', 'screen_time_before_bed_mins', 'exercise_day', 'steps_that_day', 'nap_duration_mins', 'stress_score', 'work_hours_that_day', 'chronotype', 'mental_health_condition', 'heart_rate_resting_bpm', 'sleep_aid_used', 'shift_work', 'room_temperature_celsius', 'weekend_sleep_diff_hrs', 'season', 'day_type', 'cognitive_performance_score', 'sleep_disorder_risk', 'felt_rested']


In [5]:
categorical_cols = ['gender', 'occupation', 'country', 'chronotype', 'mental_health_condition', 'season', 'day_type']
for col in categorical_cols:
    print(f"Column '{col}': {df[col].nunique()} unique values")

Column 'gender': 3 unique values
Column 'occupation': 12 unique values
Column 'country': 15 unique values
Column 'chronotype': 3 unique values
Column 'mental_health_condition': 4 unique values
Column 'season': 4 unique values
Column 'day_type': 2 unique values


**One hot encoding for everything but country, occupation, and sleep disorder risk**

In [6]:
df= pd.get_dummies(df, columns=['gender', 'chronotype', 'mental_health_condition', 'season', 'day_type'],dtype=int )
from sklearn.preprocessing import LabelEncoder
le_country = LabelEncoder()
le_occupation = LabelEncoder()

df['country_encoded'] = le_country.fit_transform(df['country'])
df['occupation_encoded'] = le_occupation.fit_transform(df['occupation'])

#  Drop the original 'country' and 'occupation' columns
df = df.drop(columns=['country', 'occupation'])

le_target= LabelEncoder()
df['sleep_disorder_risk_encoded'] = le_target.fit_transform(df['sleep_disorder_risk'])
df = df.drop(columns=['sleep_disorder_risk'])

print("Shape after encoding:", df.shape)
print("First 5 rows after encoding:")
print(df.head())

Shape after encoding: (100000, 42)
First 5 rows after encoding:
   age   bmi  sleep_duration_hrs  sleep_quality_score  rem_percentage  \
0   29  25.7                6.19                  6.6            22.5   
1   55  22.0                8.32                  6.9            26.9   
2   42  25.0                3.74                  1.0            20.2   
3   37  29.5                6.79                  6.4            17.7   
4   23  23.6                5.02                  3.2            23.3   

   deep_sleep_percentage  sleep_latency_mins  wake_episodes_per_night  \
0                   19.3                  16                        3   
1                   14.9                  17                        4   
2                   16.2                  26                        4   
3                   17.7                  13                        4   
4                   18.3                  30                        5   

   caffeine_mg_before_bed  alcohol_units_before_bed  ...  

**Training the model**

In [7]:
from sklearn.model_selection import train_test_split
X= df.drop(columns=['sleep_disorder_risk_encoded'])
y= df['sleep_disorder_risk_encoded']
X_train, X_test, y_train, y_test= train_test_split(X, y, train_size=0.8, test_size=0.2, random_state=42)
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(80000, 41)
(20000, 41)
(80000,)
(20000,)


In [8]:
from sklearn.ensemble import RandomForestClassifier
model= RandomForestClassifier(n_estimators=100, random_state=42,)
model.fit(X_train, y_train)
y_pred=model.predict(X_test)
le_target.inverse_transform(y_pred)
# model.score(X_test, y_test)

array(['Mild', 'Severe', 'Mild', ..., 'Mild', 'Healthy', 'Mild'],
      dtype=object)

In [9]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Confusion Matrix:
[[10594   298     0     0]
 [  429  6064   180     0]
 [    0   648   878   100]
 [    0     8   237   564]]
Accuracy: 0.905
              precision    recall  f1-score   support

           0       0.96      0.97      0.97     10892
           1       0.86      0.91      0.89      6673
           2       0.68      0.54      0.60      1626
           3       0.85      0.70      0.77       809

    accuracy                           0.91     20000
   macro avg       0.84      0.78      0.80     20000
weighted avg       0.90      0.91      0.90     20000



In [10]:
importances = pd.Series(model.feature_importances_, index=X_train.columns).sort_values(ascending=False).head(10)
print(importances)

sleep_duration_hrs                 0.113354
sleep_quality_score                0.109658
cognitive_performance_score        0.106467
bmi                                0.088405
mental_health_condition_Healthy    0.073778
stress_score                       0.063090
wake_episodes_per_night            0.056666
sleep_latency_mins                 0.053810
rem_percentage                     0.024276
age                                0.019851
dtype: float64


In [11]:
important_features=importances[importances>0.02].index
X_train_filtered = X_train[important_features]
X_test_filtered = X_test[important_features]
X_train_filtered.shape

# Training a new model with fewer features to compare accuracy
model_filtered= RandomForestClassifier(n_estimators=100, random_state=42,)
model_filtered.fit(X_train_filtered, y_train)
y_pred_filtered=model_filtered.predict(X_test_filtered)


In [12]:
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_filtered))
print("Accuracy:", accuracy_score(y_test, y_pred_filtered))
print(classification_report(y_test, y_pred_filtered))

Confusion Matrix:
[[10551   341     0     0]
 [  587  5772   309     5]
 [    0   528   955   143]
 [    0    16   252   541]]
Accuracy: 0.89095
              precision    recall  f1-score   support

           0       0.95      0.97      0.96     10892
           1       0.87      0.86      0.87      6673
           2       0.63      0.59      0.61      1626
           3       0.79      0.67      0.72       809

    accuracy                           0.89     20000
   macro avg       0.81      0.77      0.79     20000
weighted avg       0.89      0.89      0.89     20000



**Using XG Boost instead of Random Forest**

In [13]:
from xgboost import XGBClassifier
n_estimators=[50, 100, 200, 300] #using multiple estimators to check which yields best reults (100 seems best)
for i in n_estimators:
  print(f"Iteration number {i} ")
  xg_model= XGBClassifier(n_estimators=i, random_state=42, eval_metric='mlogloss')
  xg_model.fit(X_train_filtered, y_train)
  y_pred_xg=xg_model.predict(X_test_filtered)

  #Getting reports
  print("Confusion Matrix:")
  print(confusion_matrix(y_test, y_pred_xg))
  print("Accuracy:", accuracy_score(y_test, y_pred_xg))
  print(classification_report(y_test, y_pred_xg))

Iteration number 50 
Confusion Matrix:
[[10603   289     0     0]
 [  573  5768   330     2]
 [    0   461  1009   156]
 [    0     8   249   552]]
Accuracy: 0.8966
              precision    recall  f1-score   support

           0       0.95      0.97      0.96     10892
           1       0.88      0.86      0.87      6673
           2       0.64      0.62      0.63      1626
           3       0.78      0.68      0.73       809

    accuracy                           0.90     20000
   macro avg       0.81      0.79      0.80     20000
weighted avg       0.89      0.90      0.90     20000

Iteration number 100 
Confusion Matrix:
[[10598   294     0     0]
 [  562  5775   335     1]
 [    0   451  1004   171]
 [    0    11   243   555]]
Accuracy: 0.8966
              precision    recall  f1-score   support

           0       0.95      0.97      0.96     10892
           1       0.88      0.87      0.87      6673
           2       0.63      0.62      0.63      1626
           3     

**Cross Validation and training on xg_model**

In [15]:
from sklearn.model_selection import cross_val_score
import numpy as np
cv_scores= cross_val_score(xg_model, X_train_filtered, y_train, cv=5, scoring='accuracy')
print(cv_scores)
print(cv_scores.mean())
print(np.median(cv_scores))


[0.89125   0.88875   0.8911875 0.889375  0.889875 ]
0.8900875000000001
0.889875
